# CTFusionNet: a lightweight transformer-enhanced framework for simultaneous CT denoising and lesion localization

**Problem statement.** Design a lightweight transformer-enhanced framework for simultaneous CT image denoising and lesion localization, capable of capturing both local texture information and global anatomical context.

**Proposed model (CTFusionNet)**
```
512×512 low-dose CT
        │
 Shared hybrid encoder ── local: depthwise-conv blocks (texture, 512→128 px)
        │               └ global: efficient self-attention blocks (anatomy, 64 & 32 px)
        ├────────────────────────────┐
 Denoising decoder  ◄── Cross-Task ──►  Localization decoder
 (to 512 px)            Fusion ×2        (to 128 px)
        │           (1/8 and 1/4 scale)       │
 Clean CT image                     Heatmap + size + offset → bounding boxes
```
**Cross-task fusion (novelty):** at each shared scale, the localization branch sends a *lesion attention map* to the denoiser (preserve detail where lesions are), and the denoising branch sends *channel-attended, noise-suppressed features* to the localizer. Learnable gates start at zero, so the effect of fusion can be measured directly.

| Section | Content |
|---|---|
| 0 | Config |
| 1 | **Official NIH DeepLesion** download (key slices only, MD5-checked, resumable) |
| 2 | Official patient-level splits |
| 3 | Physics-based low-dose simulation + validation |
| 4 | Paired low-dose generation (resumable) |
| 3a | Model code `ctfusion.py` (shared with the demo) |
| 5 | Data loader + model check |
| 6 | Training (checkpoint every epoch, auto-resume) |
| 7 | Evaluation: PSNR / SSIM / RMSE(HU), FROC @ IoU≥0.5, per-lesion-type sensitivity, params & speed |
| 8 | Qualitative figures incl. fusion attention maps |
| 9 | Experiments & baselines (RED-CNN, two-stage), summary table, significance tests, paper figures |
| 10 | **Streamlit demo** (public link from Colab) |

**Run on Colab:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all` (data and results live in Google Drive).
**Run on Kaggle:** Settings → Accelerator **GPU T4 ×2** or **P100**, **Internet on**, then *Run all*. Data and results go to `/kaggle/working` — click **Save Version** to keep them. To reuse data already generated on Colab, zip the Drive `CT_Multitask_Research/data` folder, upload it as a Kaggle Dataset and add it to the notebook; it is copied in automatically.
After any disconnect, *Run all* again; finished work is skipped.
**Demo only (model already trained):** run section 0 (both cells), 3a (both cells) and 10.

**Dataset:** NIH DeepLesion — Yan K, Wang X, Lu L, Summers RM. *J. Med. Imaging* 5(3):036501, 2018. Low-dose images are **simulated**, not real low-dose acquisitions.

## 0. Configuration

In [ ]:
# ============================== CONFIG ==============================
# --- data ---
N_ZIPS     = 10         # official zips to use (of 56). 10 ≈ 5-6k key slices ≈ 5 GB (conference scale)
DROP_NOISY = False      # drop lesions flagged 'Possibly_noisy'

# --- low-dose simulation (nominal, not scanner-calibrated) ---
MU_WATER    = 0.02                             # mm^-1
N_ANGLES    = 360                              # parallel-beam views over 180°
DOSE_LEVELS = {'low': 1e6, 'moderate': 3e6}    # incident photons per detector bin
SIM_BACKEND = 'auto'    # 'auto' = GPU if it matches the skimage reference in section 3b, else CPU

# --- experiment (see the table in section 9) ---
EXPERIMENT      = 'ctfusion_full'
SEED            = 42                # key experiments: 42, 43, 44
MODEL           = 'ctfusion'        # 'ctfusion' or 'redcnn' (published denoising baseline)
USE_TRANSFORMER = True              # global self-attention blocks in the encoder
USE_FUSION      = True              # bidirectional cross-task fusion
LAMBDA_DEN      = 1.0               # denoising loss weight    (0 → localization only)
LAMBDA_DET      = 1.0               # localization loss weight (0 → denoising only)
TRAIN_ON_CLEAN  = False             # True only for 'detect_clean' (detector of the two-stage baseline)
BASE_CH         = 32

# --- training ---
EPOCHS, BATCH_SIZE, LR = 40, 8, 3e-4
RUN_NAME = f'{EXPERIMENT}_s{SEED}'
# ====================================================================

In [ ]:
import os, re, sys, json, time, glob, zlib, shutil, hashlib, zipfile, urllib.request, subprocess
ON_KAGGLE = os.path.exists('/kaggle/working')
if ON_KAGGLE:
    PROJECT = '/kaggle/working/CT_Multitask_Research'
    SCRATCH = '/kaggle/temp' if os.path.isdir('/kaggle/temp') else '/tmp'
    os.makedirs(f'{PROJECT}/data', exist_ok=True)
    # Reuse earlier work added as notebook input (a previous version's output or a Kaggle Dataset):
    # every folder that contains data/DL_info.csv is merged in (data, low-dose images, checkpoints, results).
    for found in sorted(glob.glob('/kaggle/input/**/data/DL_info.csv', recursive=True)):
        src = os.path.dirname(os.path.dirname(found))
        print('Restoring previous work from', src)
        shutil.copytree(src, PROJECT, dirs_exist_ok=True)
    ck = glob.glob(f'{PROJECT}/models/*/last.pt')
    print('Checkpoints restored:', [p.split('/')[-2] for p in ck] or 'none')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = '/content/drive/MyDrive/CT_Multitask_Research'
    SCRATCH = '/content'
print('Platform:', 'Kaggle' if ON_KAGGLE else 'Colab', '| project folder:', PROJECT)

import numpy as np, pandas as pd, cv2, matplotlib.pyplot as plt, requests
from tqdm.auto import tqdm
from joblib import Parallel, delayed
try:
    requests.head('https://huggingface.co', timeout=10); print('Internet: OK')
except Exception:
    print('Internet: NOT available' + (' — turn on "Internet" in the Kaggle notebook settings' if ON_KAGGLE else ''))

DATA, IMG_DIR = f'{PROJECT}/data', f'{PROJECT}/data/images'
SPLIT_DIR, LD_DIR = f'{DATA}/splits', f'{DATA}/lowdose'
MODEL_DIR, RES_DIR = f'{PROJECT}/models/{RUN_NAME}', f'{PROJECT}/results/{RUN_NAME}'
for d in [IMG_DIR, SPLIT_DIR, MODEL_DIR, RES_DIR, f'{PROJECT}/code', f'{PROJECT}/results/figures'] + [f'{LD_DIR}/{k}' for k in DOSE_LEVELS]:
    os.makedirs(d, exist_ok=True)
LOCAL_CACHE = not ON_KAGGLE        # Kaggle storage is already local; Drive is slow so Colab caches
CONFIG = {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, bool, dict))
          and k not in ('PROJECT', 'SCRATCH', 'DATA', 'IMG_DIR', 'SPLIT_DIR', 'LD_DIR', 'MODEL_DIR', 'RES_DIR')}
print(json.dumps(CONFIG, indent=1))

## 1. Official DeepLesion download
* Image zips come **from NIH's own Box links** (`nihcc.box.com`), parsed from NIH's official `batch_download_zips.py`, and are verified against NIH's MD5 checksums.
* `DL_info.csv` and that script are read from the public Hugging Face mirror of the official release (`farrell236/DeepLesion`).
* Only annotated key slices are kept; each zip is deleted after extraction. `.done` markers make this resumable.

In [ ]:
MIRROR = 'https://huggingface.co/datasets/farrell236/DeepLesion/resolve/main'
DL_INFO_PATH = f'{DATA}/DL_info.csv'
if not os.path.exists(DL_INFO_PATH):
    urllib.request.urlretrieve(f'{MIRROR}/DL_info.csv', DL_INFO_PATH)
dl = pd.read_csv(DL_INFO_PATH)
print('DL_info rows:', len(dl))

pending = [k for k in range(1, N_ZIPS + 1) if not os.path.exists(f'{IMG_DIR}/zip{k:02d}/.done')]
ZIP_LINKS, MD5 = [], {}
if pending:
    script = requests.get(f'{MIRROR}/scripts/batch_download_zips.py', timeout=60).text
    ZIP_LINKS = re.findall(r"https://nihcc\.box\.com/shared/static/\w+\.zip", script)
    MD5_LINKS = re.findall(r"https://nihcc\.box\.com/shared/static/\w+\.txt", script)
    assert len(ZIP_LINKS) == 56, f'Expected 56 official zip links, found {len(ZIP_LINKS)}'
    try:
        MD5 = {f: h for h, f in re.findall(r'([0-9a-fA-F]{32})\s+\*?(\S+\.zip)', requests.get(MD5_LINKS[0], timeout=60).text)}
    except Exception as e:
        print('MD5 list unavailable, zips will only be structure-checked:', e)
print(f'Zips still to process: {pending or "none"}')

In [ ]:
KEY_NAMES = set(dl['File_name'])
TMP = f'{SCRATCH}/tmp_zip'; os.makedirs(TMP, exist_ok=True)
total, used, free = shutil.disk_usage(PROJECT)
need = 0.5 * len(pending)
print(f'Free space at project folder: {free / 2**30:.1f} GB | estimated need (slices + low-dose): ~{need:.1f} GB')
if need > free / 2**30: print('WARNING: not enough space — lower N_ZIPS or free up space.')

def existing_key_slices():
    return {os.path.basename(p) for p in glob.glob(f'{IMG_DIR}/**/*.png', recursive=True)}

def download(url, path, retries=3):
    for attempt in range(1, retries + 1):
        try:
            with requests.get(url, stream=True, timeout=120) as r:
                r.raise_for_status()
                with open(path, 'wb') as f, tqdm(total=int(r.headers.get('content-length', 0)), unit='B',
                                                unit_scale=True, desc=os.path.basename(path)) as bar:
                    for chunk in r.iter_content(8 << 20):
                        f.write(chunk); bar.update(len(chunk))
            if zipfile.is_zipfile(path): return True
            print('  not a valid zip, retrying')
        except Exception as e:
            print(f'  attempt {attempt} failed: {e}')
        time.sleep(10)
    return False

def md5sum(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(16 << 20), b''): h.update(chunk)
    return h.hexdigest()

for k in pending:
    zname, outdir = f'Images_png_{k:02d}.zip', f'{IMG_DIR}/zip{k:02d}'
    os.makedirs(outdir, exist_ok=True); zpath = f'{TMP}/{zname}'
    print(f'=== {zname} ===')
    if not download(ZIP_LINKS[k - 1], zpath):
        print(f'{zname}: download failed, rerun this cell later'); continue
    if zname in MD5 and md5sum(zpath) != MD5[zname].lower():
        print('MD5 mismatch, deleting'); os.remove(zpath); continue
    have, n_new = existing_key_slices(), 0
    with zipfile.ZipFile(zpath) as z:
        for member in z.namelist():
            parts = member.replace('\\', '/').split('/')
            if len(parts) < 2 or not parts[-1].endswith('.png'): continue
            name = f'{parts[-2]}_{parts[-1]}'            # folder_slice.png == DL_info File_name
            if name in KEY_NAMES and name not in have:
                with open(f'{outdir}/{name}', 'wb') as f: f.write(z.read(member))
                n_new += 1
    os.remove(zpath); open(f'{outdir}/.done', 'w').write('ok')
    print(f'{zname}: {n_new} key slices saved')
print('Key slices available:', len(existing_key_slices()))

## 2. Official patient-level splits

In [ ]:
path_map = {os.path.basename(p): p for p in glob.glob(f'{IMG_DIR}/**/*.png', recursive=True)}
df = dl[dl['File_name'].isin(path_map)].copy()
if DROP_NOISY: df = df[df['Possibly_noisy'] == 0]
n0 = df.File_name.nunique()
df = df[df['Image_size'].astype(str).str.split(',').str[0].astype(int) == 512]   # model input is 512x512
print(f'Kept {df.File_name.nunique()} of {n0} slices (dropped {n0 - df.File_name.nunique()} non-512x512 slices)')
df['Image_Path'] = df['File_name'].map(path_map)
df['Spacing'] = df['Spacing_mm_px_'].str.split(',').str[0].astype(float)
stats = []
for name, k in [('train', 1), ('val', 2), ('test', 3)]:
    s = df[df['Train_Val_Test'] == k]; s.to_csv(f'{SPLIT_DIR}/{name}_df.csv', index=False)
    stats.append({'split': name, 'lesions': len(s), 'images': s.File_name.nunique(), 'patients': s.Patient_index.nunique()})
stats = pd.DataFrame(stats); print(stats.to_string(index=False))
stats.to_csv(f'{PROJECT}/results/dataset_stats.csv', index=False)
P = [set(df.loc[df.Train_Val_Test == k, 'Patient_index']) for k in (1, 2, 3)]
print('Patient overlap (should be 0 0 0):', len(P[0] & P[1]), len(P[0] & P[2]), len(P[1] & P[2]))
images = df.drop_duplicates('File_name').reset_index(drop=True)

## 3. Low-dose simulation & validation
HU → μ → Radon → N ~ Poisson(I₀e^(−p)) → −ln(N/I₀) → ramp-filtered back projection → HU.

### 3a. Model & simulation code (written to `ctfusion.py`, also used by the Streamlit app)

In [ ]:
%%writefile ctfusion.py
"""CTFusionNet: lightweight hybrid CNN-Transformer with bidirectional cross-task fusion
for simultaneous low-dose CT denoising and lesion localization."""
import numpy as np, cv2, torch, torch.nn as nn, torch.nn.functional as F
from skimage.transform import radon, iradon

HU_MIN, HU_MAX = -1024, 3071      # HU range used for normalisation
STRIDE  = 4                       # detection output stride (512 -> 128)
MAX_OBJ = 8                       # max lesions per slice (DeepLesion has 1-3)
WINDOWS = {'Soft tissue': (-160, 240), 'Liver': (-45, 205), 'Lung': (-1350, 150), 'Bone': (-450, 1050)}

# ------------------------------------------------------------------ I/O
def hu_from_png_bytes(b):
    img = cv2.imdecode(np.frombuffer(b, np.uint8), cv2.IMREAD_UNCHANGED)
    if img is None: raise ValueError('Could not decode image')
    if img.ndim == 3: img = img[..., 0]
    if img.dtype == np.uint16:                       # DeepLesion format: HU = pixel - 32768
        return img.astype(np.float32) - 32768, True
    return img.astype(np.float32) / 255 * 400 - 160, False   # 8-bit: assume soft-tissue window

def load_hu(path):
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None: raise FileNotFoundError(path)
    return img.astype(np.float32) - 32768

def save_hu_png(hu, path):
    cv2.imwrite(path, np.clip(np.round(hu) + 32768, 0, 65535).astype(np.uint16))

def norm(hu):  return (np.clip(hu, HU_MIN, HU_MAX) - HU_MIN) / (HU_MAX - HU_MIN)
def to_hu(x):  return x * (HU_MAX - HU_MIN) + HU_MIN
def window(hu, win=(-160, 240)):
    return (np.clip((hu - win[0]) / (win[1] - win[0]), 0, 1) * 255).astype(np.uint8)

# ------------------------------------------------- low-dose simulation
def project(hu, spacing, n_angles=360, mu_water=0.02):
    theta = np.linspace(0., 180., n_angles, endpoint=False)
    mu = np.clip(mu_water * (1 + hu / 1000.0), 0, None)
    return radon(mu, theta=theta, circle=False) * spacing, theta

def fbp(sino, theta, spacing, n, mu_water=0.02):
    mu = iradon(sino / spacing, theta=theta, filter_name='ramp', circle=False, output_size=n)
    return (mu / mu_water - 1) * 1000.0

def add_poisson(sino, I0, seed):
    rng = np.random.default_rng(seed)
    cnt = np.maximum(rng.poisson(I0 * np.exp(-sino)), 1)
    return -np.log(cnt / I0)

def simulate_low_dose(hu, spacing, I0, seed=0, n_angles=360, mu_water=0.02):
    s, th = project(hu, spacing, n_angles, mu_water)
    return fbp(add_poisson(s, I0, seed), th, spacing, hu.shape[0], mu_water)

class TorchCT:
    """Same physics on the GPU: parallel-beam Radon (bilinear line sampling, 1-px steps),
    Ram-Lak ramp filter (skimage construction), linear-interpolated back projection, pi/(2A) scaling.
    Validated against the skimage version in the notebook before use."""
    def __init__(self, n=512, n_angles=360, device='cuda', chunk=30):
        self.n, self.A, self.dev, self.chunk = n, n_angles, device, chunk
        self.D = int(np.ceil(np.sqrt(2) * n))
        th = torch.deg2rad(torch.linspace(0, 180, n_angles + 1, device=device)[:-1])
        self.cos, self.sin = torch.cos(th), torch.sin(th)
        self.t = torch.arange(self.D, device=device, dtype=torch.float32) - (self.D - 1) / 2
        g = torch.arange(n, device=device, dtype=torch.float32) - (n - 1) / 2
        self.yy, self.xx = torch.meshgrid(g, g, indexing='ij')
        self.size = max(64, int(2 ** np.ceil(np.log2(2 * self.D))))
        k = np.concatenate((np.arange(1, self.size / 2 + 1, 2, dtype=int), np.arange(self.size / 2 - 1, 0, -2, dtype=int)))
        f = np.zeros(self.size); f[0] = 0.25; f[1::2] = -1 / (np.pi * k) ** 2
        self.filt = torch.tensor(2 * np.real(np.fft.fft(f)), dtype=torch.float32, device=device)

    def radon(self, img):                                   # img (N,N) -> (A,D) line integrals in pixel units
        x, half, out = img[None, None], (self.n - 1) / 2, []
        for a0 in range(0, self.A, self.chunk):
            cs, sn = self.cos[a0:a0 + self.chunk, None, None], self.sin[a0:a0 + self.chunk, None, None]
            t, s = self.t[None, :, None], self.t[None, None, :]
            px, py = t * cs - s * sn, t * sn + s * cs        # points on each ray
            grid = torch.stack([px / half, py / half], -1).reshape(1, -1, self.D, 2)
            v = F.grid_sample(x, grid, mode='bilinear', padding_mode='zeros', align_corners=True)
            out.append(v.view(-1, self.D, self.D).sum(-1))   # integrate along each ray
        return torch.cat(out, 0)

    def fbp(self, sino):                                    # sino (A,D) -> (N,N)
        q = torch.fft.ifft(torch.fft.fft(sino, n=self.size, dim=-1) * self.filt, dim=-1).real[:, :self.D]
        half_d, rec = (self.D - 1) / 2, torch.zeros(self.n * self.n, device=self.dev)
        for a0 in range(0, self.A, self.chunk):
            cs, sn = self.cos[a0:a0 + self.chunk, None, None], self.sin[a0:a0 + self.chunk, None, None]
            pos = (self.xx[None] * cs + self.yy[None] * sn + half_d).reshape(cs.shape[0], -1)
            i0 = pos.floor().long(); w = pos - i0
            valid = (i0 >= 0) & (i0 < self.D - 1); i0 = i0.clamp(0, self.D - 2)
            qa = q[a0:a0 + self.chunk]
            v = qa.gather(1, i0) * (1 - w) + qa.gather(1, i0 + 1) * w
            rec += (v * valid).sum(0)
        return rec.view(self.n, self.n) * np.pi / (2 * self.A)

    @torch.no_grad()
    def simulate(self, hu, spacing, I0s, seed, mu_water=0.02, return_clean=False):
        """hu: (N,N) numpy HU. Returns list of low-dose HU images (numpy), one per I0."""
        h = torch.as_tensor(hu, dtype=torch.float32, device=self.dev)
        p = self.radon((mu_water * (1 + h / 1000)).clamp(min=0)) * spacing
        to_hu_ = lambda s: ((self.fbp(s / spacing) / mu_water - 1) * 1000).cpu().numpy()
        outs = []
        for k, I0 in enumerate(I0s):
            g = torch.Generator(device=self.dev); g.manual_seed(int(seed) + k)
            cnt = torch.poisson(I0 * torch.exp(-p), generator=g).clamp(min=1)
            outs.append(to_hu_(-torch.log(cnt / I0)))
        return (outs, to_hu_(p)) if return_clean else outs

# ------------------------------------------------------ detection targets
def make_targets(boxes, size=512):
    H = size // STRIDE
    hm = np.zeros((H, H), np.float32)
    wh, off = np.zeros((2, MAX_OBJ), np.float32), np.zeros((2, MAX_OBJ), np.float32)
    ind, mask = np.zeros(MAX_OBJ, np.int64), np.zeros(MAX_OBJ, np.float32)
    yy, xx = np.mgrid[0:H, 0:H]
    for k, (x1, y1, x2, y2) in enumerate(list(boxes)[:MAX_OBJ]):
        cx, cy = (x1 + x2) / 2 / STRIDE, (y1 + y2) / 2 / STRIDE
        ix, iy = int(np.clip(cx, 0, H - 1)), int(np.clip(cy, 0, H - 1))
        w, h = (x2 - x1) / STRIDE, (y2 - y1) / STRIDE
        sigma = max(w, h) / 6 + 1
        hm = np.maximum(hm, np.exp(-((xx - ix) ** 2 + (yy - iy) ** 2) / (2 * sigma ** 2)))
        hm[iy, ix] = 1.0
        wh[:, k], off[:, k] = (w, h), (cx - ix, cy - iy)
        ind[k], mask[k] = iy * H + ix, 1
    return hm, wh, off, ind, mask

# ---------------------------------------------------------------- blocks
class LocalBlock(nn.Module):
    """Local texture: depthwise 7x7 conv + pointwise MLP (ConvNeXt-lite), residual."""
    def __init__(self, c, exp=2):
        super().__init__()
        self.dw, self.bn = nn.Conv2d(c, c, 7, padding=3, groups=c), nn.BatchNorm2d(c)
        self.pw1, self.act, self.pw2 = nn.Conv2d(c, c * exp, 1), nn.GELU(), nn.Conv2d(c * exp, c, 1)
    def forward(self, x):
        return x + self.pw2(self.act(self.pw1(self.bn(self.dw(x)))))

class GlobalBlock(nn.Module):
    """Global anatomical context: efficient multi-head self-attention
    (spatial-reduction keys/values, conditional positional encoding) + MLP."""
    def __init__(self, c, heads=4, sr=1, mlp=2):
        super().__init__()
        self.h = heads
        self.pos = nn.Conv2d(c, c, 3, padding=1, groups=c)
        self.n1, self.n2 = nn.LayerNorm(c), nn.LayerNorm(c)
        self.q, self.kv, self.proj = nn.Linear(c, c), nn.Linear(c, 2 * c), nn.Linear(c, c)
        self.sr = nn.Sequential(nn.Conv2d(c, c, sr, stride=sr)) if sr > 1 else None
        self.srn = nn.LayerNorm(c) if sr > 1 else None
        self.mlp = nn.Sequential(nn.Linear(c, c * mlp), nn.GELU(), nn.Linear(c * mlp, c))
    def forward(self, x):
        B, C, H, W = x.shape
        x = x + self.pos(x)
        t = x.flatten(2).transpose(1, 2)                         # B, N, C
        h = self.n1(t)
        q = self.q(h).view(B, -1, self.h, C // self.h).transpose(1, 2)
        if self.sr is not None:
            kin = self.srn(self.sr(h.transpose(1, 2).reshape(B, C, H, W)).flatten(2).transpose(1, 2))
        else:
            kin = h
        k, v = self.kv(kin).view(B, -1, 2, self.h, C // self.h).permute(2, 0, 3, 1, 4)
        a = F.scaled_dot_product_attention(q, k, v)              # B, heads, N, d
        t = t + self.proj(a.transpose(1, 2).reshape(B, -1, C))
        t = t + self.mlp(self.n2(t))
        return t.transpose(1, 2).reshape(B, C, H, W)

class Up(nn.Module):
    def __init__(self, cin, cskip, cout):
        super().__init__()
        self.reduce, self.bn, self.block = nn.Conv2d(cin + cskip, cout, 1), nn.BatchNorm2d(cout), LocalBlock(cout)
    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False)
        return self.block(F.gelu(self.bn(self.reduce(torch.cat([x, skip], 1)))))

class CrossTaskFusion(nn.Module):
    """Bidirectional cross-task fusion (CTF).
    det -> den : lesion-aware spatial attention tells the denoiser WHERE detail must be preserved.
    den -> det : channel attention from denoising features passes noise-suppressed evidence to detection.
    Learnable gates start at 0, so training starts from the no-fusion model."""
    def __init__(self, c):
        super().__init__()
        self.lesion_att = nn.Sequential(nn.Conv2d(c, c // 4, 1), nn.GELU(), nn.Conv2d(c // 4, 1, 3, padding=1))
        self.det2den = nn.Sequential(nn.Conv2d(2 * c, c, 1), nn.BatchNorm2d(c), nn.GELU())
        self.ch_att = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(c, c // 4, 1), nn.GELU(), nn.Conv2d(c // 4, c, 1))
        self.den2det = nn.Sequential(nn.Conv2d(2 * c, c, 1), nn.BatchNorm2d(c), nn.GELU())
        self.g_den, self.g_det = nn.Parameter(torch.zeros(1)), nn.Parameter(torch.zeros(1))
    def forward(self, f_den, f_det):
        a  = torch.sigmoid(self.lesion_att(f_det))                # B,1,H,W lesion attention
        cw = torch.sigmoid(self.ch_att(f_den))                    # B,C,1,1 channel weights
        den = f_den + self.g_den * self.det2den(torch.cat([f_den, f_det], 1)) * a
        det = f_det + self.g_det * self.den2det(torch.cat([f_det, f_den], 1)) * cw
        return den, det, a

def _head(cin, cout, bias=0.0):
    h = nn.Sequential(nn.Conv2d(cin, 64, 3, padding=1), nn.ReLU(inplace=True), nn.Conv2d(64, cout, 1))
    nn.init.constant_(h[-1].bias, bias)
    return h

class CTFusionNet(nn.Module):
    """Shared hybrid encoder -> (denoising decoder, localization decoder) with CTF at 1/8 and 1/4 scale."""
    def __init__(self, base=32, use_transformer=True, use_fusion=True):
        super().__init__()
        ch = [base, base * 2, base * 4, base * 6, base * 8]       # 512, 256, 128, 64, 32 px
        self.stem = nn.Sequential(nn.Conv2d(1, ch[0], 3, padding=1), nn.BatchNorm2d(ch[0]), nn.GELU(), LocalBlock(ch[0]))
        self.down = nn.ModuleList([nn.Sequential(nn.BatchNorm2d(ch[i]), nn.Conv2d(ch[i], ch[i + 1], 2, stride=2))
                                   for i in range(4)])
        g3 = GlobalBlock(ch[3], sr=2) if use_transformer else LocalBlock(ch[3])
        g4 = [GlobalBlock(ch[4]), GlobalBlock(ch[4])] if use_transformer else [LocalBlock(ch[4]), LocalBlock(ch[4])]
        self.stages = nn.ModuleList([
            nn.Sequential(LocalBlock(ch[1]), LocalBlock(ch[1])),
            nn.Sequential(LocalBlock(ch[2]), LocalBlock(ch[2])),
            nn.Sequential(LocalBlock(ch[3]), g3),
            nn.Sequential(*g4)])
        self.den_up = nn.ModuleList([Up(ch[4], ch[3], ch[3]), Up(ch[3], ch[2], ch[2]),
                                     Up(ch[2], ch[1], ch[1]), Up(ch[1], ch[0], ch[0])])
        self.det_up = nn.ModuleList([Up(ch[4], ch[3], ch[3]), Up(ch[3], ch[2], ch[2])])
        self.fuse = nn.ModuleList([CrossTaskFusion(ch[3]), CrossTaskFusion(ch[2])]) if use_fusion else None
        self.den_head = nn.Conv2d(ch[0], 1, 3, padding=1)
        self.hm, self.wh, self.off = _head(ch[2], 1, -2.19), _head(ch[2], 2), _head(ch[2], 2)

    def forward(self, x):
        f = [self.stem(x)]
        for d, s in zip(self.down, self.stages):
            f.append(s(d(f[-1])))
        den = det = f[4]
        atts = []
        for i in range(2):                                        # 1/8 and 1/4 scale: both decoders + fusion
            den, det = self.den_up[i](den, f[3 - i]), self.det_up[i](det, f[3 - i])
            if self.fuse is not None:
                den, det, a = self.fuse[i](den, det); atts.append(a)
        for i in (2, 3):                                          # denoising continues to full resolution
            den = self.den_up[i](den, f[3 - i])
        return {'den': x + self.den_head(den), 'hm': self.hm(det), 'wh': self.wh(det),
                'off': self.off(det), 'att': atts}

# ---------------------------------------------------------------- losses
def gather_feat(feat, ind):                     # feat B,C,H,W ; ind B,M  ->  B,C,M
    B, C = feat.shape[:2]
    return feat.reshape(B, C, -1).gather(2, ind.unsqueeze(1).expand(B, C, ind.shape[1]))

def focal_loss(logits, gt):
    p = torch.sigmoid(logits.float()).clamp(1e-4, 1 - 1e-4)
    pos = (gt >= 0.999).float()
    loss = -(pos * (1 - p) ** 2 * torch.log(p) + (1 - pos) * (1 - gt) ** 4 * p ** 2 * torch.log(1 - p)).sum()
    return loss / pos.sum().clamp(min=1)

def detection_loss(out, hm, wh, off, ind, mask):
    m = mask.unsqueeze(1)
    n = mask.sum().clamp(min=1)
    l_hm  = focal_loss(out['hm'], hm)
    l_wh  = (torch.abs(gather_feat(out['wh'].float(), ind) - wh) * m).sum() / n
    l_off = (torch.abs(gather_feat(out['off'].float(), ind) - off) * m).sum() / n
    return l_hm + 0.1 * l_wh + l_off, (l_hm.item(), l_wh.item(), l_off.item())

# ---------------------------------------------------------------- decoding
@torch.no_grad()
def decode(out, K=20, thr=0.01):
    """Returns, per image, a list of (score, [x1, y1, x2, y2]) sorted by score."""
    p = torch.sigmoid(out['hm'].float())
    p = p * (p == F.max_pool2d(p, 3, 1, 1))
    B, _, H, W = p.shape
    scores, inds = p.view(B, -1).topk(K)
    ys, xs = (inds // W).float(), (inds % W).float()
    wh, off = gather_feat(out['wh'].float(), inds), gather_feat(out['off'].float(), inds)
    cx, cy = (xs + off[:, 0]) * STRIDE, (ys + off[:, 1]) * STRIDE
    w, h = wh[:, 0].clamp(min=1) * STRIDE, wh[:, 1].clamp(min=1) * STRIDE
    boxes = torch.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], -1).cpu().numpy()
    scores = scores.cpu().numpy()
    return [[(float(s), b.tolist()) for s, b in zip(scores[i], boxes[i]) if s >= thr] for i in range(B)]

def iou(box, boxes):
    boxes = np.asarray(boxes, np.float32).reshape(-1, 4)
    x1, y1 = np.maximum(box[0], boxes[:, 0]), np.maximum(box[1], boxes[:, 1])
    x2, y2 = np.minimum(box[2], boxes[:, 2]), np.minimum(box[3], boxes[:, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    a = (box[2] - box[0]) * (box[3] - box[1]); b = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    return inter / np.maximum(a + b - inter, 1e-6)

def match_detections(dets, gts, iou_thr=0.5):
    """Greedy matching. Returns (records [(score, is_tp)], lesion_scores per GT (-1 = missed))."""
    recs, les = [], []
    for d, g in zip(dets, gts):
        g = np.asarray(g, np.float32).reshape(-1, 4); used = {}
        for s, b in d:
            ov = iou(b, g) if len(g) else np.zeros(0)
            for j in used: ov[j] = 0
            j = int(ov.argmax()) if len(ov) else -1
            if j >= 0 and ov[j] >= iou_thr:
                used[j] = s; recs.append((s, True))
            else:
                recs.append((s, False))
        les.append([used.get(j, -1.0) for j in range(len(g))])
    return recs, les

def froc(recs, n_images, n_lesions, fps=(0.5, 1, 2, 4, 8)):
    recs = sorted(recs, key=lambda r: -r[0])
    tp = np.cumsum([r[1] for r in recs]); fp = np.cumsum([not r[1] for r in recs])
    res, thr = {}, {}
    for f in fps:
        ok = np.where(fp <= f * n_images)[0]
        res[f'sens@{f}FP'] = float(tp[ok[-1]] / n_lesions) if len(ok) else 0.0
        thr[f] = recs[ok[-1]][0] if len(ok) else 1.0
    return res, thr

def froc_curve(recs, n_images, n_lesions, max_fp=10, n=200):
    recs = sorted(recs, key=lambda r: -r[0])
    tp = np.cumsum([r[1] for r in recs]); fp = np.cumsum([not r[1] for r in recs])
    xs = np.linspace(0.125, max_fp, n); ys = []
    for f in xs:
        ok = np.where(fp <= f * n_images)[0]
        ys.append(tp[ok[-1]] / n_lesions if len(ok) else 0.0)
    return xs, np.array(ys)

class REDCNN(nn.Module):
    """RED-CNN baseline (Chen et al., IEEE TMI 2017): 5 conv + 5 deconv with residual shortcuts."""
    def __init__(self, c=96):
        super().__init__()
        self.conv = nn.ModuleList([nn.Conv2d(1, c, 5)] + [nn.Conv2d(c, c, 5) for _ in range(4)])
        self.tconv = nn.ModuleList([nn.ConvTranspose2d(c, c, 5) for _ in range(4)] + [nn.ConvTranspose2d(c, 1, 5)])
    def forward(self, x):
        r1 = x
        o = F.relu(self.conv[0](x)); o = F.relu(self.conv[1](o)); r2 = o
        o = F.relu(self.conv[2](o)); o = F.relu(self.conv[3](o)); r3 = o
        o = F.relu(self.conv[4](o))
        o = self.tconv[0](o) + r3
        o = self.tconv[2](F.relu(self.tconv[1](F.relu(o)))) + r2
        o = self.tconv[4](F.relu(self.tconv[3](F.relu(o)))) + r1
        return {'den': F.relu(o), 'att': []}

def build_model(cfg):
    if cfg.get('MODEL', 'ctfusion') == 'redcnn':
        return REDCNN()
    return CTFusionNet(cfg['BASE_CH'], cfg['USE_TRANSFORMER'], cfg['USE_FUSION'])

In [ ]:
sys.path.insert(0, os.getcwd())
import importlib, ctfusion; importlib.reload(ctfusion)
from ctfusion import *
shutil.copy('ctfusion.py', f'{PROJECT}/code/ctfusion.py')
print('ctfusion.py loaded from', os.getcwd())

In [ ]:
r = images[images.Train_Val_Test == 1].iloc[0]
hu = np.clip(load_hu(r.Image_Path), HU_MIN, HU_MAX); sp = r.Spacing
sino, theta = project(hu, sp, N_ANGLES, MU_WATER)
hu_fbp = fbp(sino, theta, sp, hu.shape[0], MU_WATER); body = hu > -500
print(f'{r.File_name} | spacing {sp} mm | FBP error without noise: {np.sqrt(np.mean((hu_fbp - hu)[body] ** 2)):.1f} HU')
levels = sorted({1e5, 3e5, 1e6, 3e6, 1e7} | set(DOSE_LEVELS.values()))
rec = {I0: fbp(add_poisson(sino, I0, SEED), theta, sp, hu.shape[0], MU_WATER) for I0 in levels}
sweep = pd.DataFrame([{'I0': I0, 'added_noise_std_HU': float((rec[I0] - hu_fbp)[body].std())} for I0 in levels])
print(sweep.to_string(index=False)); sweep.to_csv(f'{PROJECT}/results/noise_sweep.csv', index=False)
fig, ax = plt.subplots(1, len(levels) + 1, figsize=(4 * (len(levels) + 1), 4))
for a, (img, t) in zip(ax, [(hu, 'Original')] + [(rec[I0], f'I0={I0:.0e}') for I0 in levels]):
    a.imshow(img, cmap='gray', vmin=-160, vmax=240); a.set_title(t); a.axis('off')
plt.tight_layout(); plt.savefig(f'{PROJECT}/results/figures/noise_sweep.png', dpi=150); plt.show()

### 3b. GPU simulation check
The GPU implementation is compared with the skimage reference on 3 training slices and is only used if the noise-free FBP error
and the added noise level agree within 15 %; otherwise generation falls back to the CPU.

In [ ]:
import torch
use_gpu_sim = False
if SIM_BACKEND in ('auto', 'gpu') and torch.cuda.is_available():
    tct = TorchCT(512, N_ANGLES, 'cuda')
    rows_chk = []
    for _, r in images[images.Train_Val_Test == 1].head(3).iterrows():
        h = np.clip(load_hu(r.Image_Path), HU_MIN, HU_MAX); body = h > -500
        s, th = project(h, r.Spacing, N_ANGLES, MU_WATER); cpu_clean = fbp(s, th, r.Spacing, 512, MU_WATER)
        cpu_ld = fbp(add_poisson(s, DOSE_LEVELS['low'], 0), th, r.Spacing, 512, MU_WATER)
        t = time.time()
        (gpu_ld,), gpu_clean = tct.simulate(h, r.Spacing, [DOSE_LEVELS['low']], 0, MU_WATER, return_clean=True)
        torch.cuda.synchronize(); ms = (time.time() - t) * 1000
        rows_chk.append({'file': r.File_name,
                         'cpu_fbp_err': np.sqrt(np.mean((cpu_clean - h)[body] ** 2)),
                         'gpu_fbp_err': np.sqrt(np.mean((gpu_clean - h)[body] ** 2)),
                         'cpu_noise': (cpu_ld - cpu_clean)[body].std(), 'gpu_noise': (gpu_ld - gpu_clean)[body].std(),
                         'gpu_ms': ms})
    chk = pd.DataFrame(rows_chk).round(2); print(chk.to_string(index=False))
    noise_ok = (abs(chk.gpu_noise / chk.cpu_noise - 1) < 0.15).all()
    fbp_ok = (chk.gpu_fbp_err < chk.cpu_fbp_err * 1.15 + 5).all()
    use_gpu_sim = bool(noise_ok and fbp_ok) or SIM_BACKEND == 'gpu'
    chk.to_csv(f'{PROJECT}/results/gpu_sim_check.csv', index=False)
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    for a, img, t in zip(ax, [h, cpu_ld, gpu_ld], ['Original', 'CPU (skimage) low-dose', 'GPU low-dose']):
        a.imshow(img, cmap='gray', vmin=-160, vmax=240); a.set_title(t); a.axis('off')
    plt.tight_layout(); plt.savefig(f'{PROJECT}/results/figures/gpu_sim_check.png', dpi=120); plt.show()
print('Simulation backend:', 'GPU' if use_gpu_sim else 'CPU (skimage)')

## 4. Generate paired low-dose images (resumable)

In [ ]:
from concurrent.futures import ThreadPoolExecutor
def process_one(row):
    fname = row['File_name']
    tg = {d: f'{LD_DIR}/{d}/{fname}' for d in DOSE_LEVELS}
    if all(os.path.exists(p) for p in tg.values()): return 'skipped'
    try:
        h = np.clip(load_hu(row['Image_Path']), HU_MIN, HU_MAX)
        s, th = project(h, row['Spacing'], N_ANGLES, MU_WATER)
        seed = zlib.crc32(fname.encode())
        for k, (d, I0) in enumerate(DOSE_LEVELS.items()):
            save_hu_png(fbp(add_poisson(s, I0, seed + k), th, row['Spacing'], h.shape[0], MU_WATER), tg[d])
        return 'ok'
    except Exception as e:
        return f'error: {e}'

todo = [r for _, r in images.iterrows() if not all(os.path.exists(f'{LD_DIR}/{d}/{r.File_name}') for d in DOSE_LEVELS)]
print(f'{len(images) - len(todo)} done, {len(todo)} to generate')
status = []
if use_gpu_sim and todo:
    def read(r): return r, np.clip(load_hu(r['Image_Path']), HU_MIN, HU_MAX)
    with ThreadPoolExecutor(8) as io:
        writes, bar = [], tqdm(total=len(todo), desc='GPU simulation')
        for c in range(0, len(todo), 64):                       # read ahead 64 slices at a time
            for r, h in io.map(read, todo[c:c + 64]):
                bar.update(1)
                try:
                    outs = tct.simulate(h, r['Spacing'], list(DOSE_LEVELS.values()), zlib.crc32(r['File_name'].encode()), MU_WATER)
                    for d, img in zip(DOSE_LEVELS, outs):
                        writes.append(io.submit(save_hu_png, img, f'{LD_DIR}/{d}/{r["File_name"]}'))
                    status.append('ok')
                except Exception as e:
                    status.append(f'error: {e}')
        for w in writes: w.result()
elif todo:
    for b in tqdm(range(0, len(todo), 40), desc='CPU batches'):
        status += Parallel(n_jobs=os.cpu_count())(delayed(process_one)(r) for r in todo[b:b + 40])
print('Errors:', [s for s in status if s.startswith('error')][:5])

df['LowDose_Path'] = f'{LD_DIR}/low/' + df['File_name']
df['ModDose_Path'] = f'{LD_DIR}/moderate/' + df['File_name']
bad = [p for p in pd.concat([df.LowDose_Path, df.ModDose_Path]).unique()
       if (im := cv2.imread(p, cv2.IMREAD_UNCHANGED)) is None or im.shape != (512, 512)]
for p in bad:
    if os.path.exists(p): os.remove(p)
print('Unreadable files removed (rerun this cell to regenerate):', len(bad))
df.to_csv(f'{SPLIT_DIR}/manifest_paired.csv', index=False)
print('manifest_paired.csv:', len(df), 'lesion rows')

## 5. Data loader + model check

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
torch.manual_seed(SEED); np.random.seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device, torch.cuda.get_device_name(0) if device == 'cuda' else '')

manifest = pd.read_csv(f'{SPLIT_DIR}/manifest_paired.csv')
if LOCAL_CACHE:
    CACHE, cols = f'{SCRATCH}/cache', ['Image_Path', 'LowDose_Path', 'ModDose_Path']
    def cp(p):
        q = p.replace(PROJECT, CACHE)
        if not os.path.exists(q): os.makedirs(os.path.dirname(q), exist_ok=True); shutil.copy(p, q)
    Parallel(n_jobs=16, prefer='threads')(delayed(cp)(p) for p in tqdm(pd.unique(manifest[cols].values.ravel()), desc='caching'))
    for c in cols: manifest[c] = manifest[c].str.replace(PROJECT, CACHE, regex=False)

grp = manifest.groupby('File_name').agg(Clean=('Image_Path', 'first'), Low=('LowDose_Path', 'first'),
        Mod=('ModDose_Path', 'first'), split=('Train_Val_Test', 'first'),
        boxes=('Bounding_boxes', list), types=('Coarse_lesion_type', list)).reset_index()
grp['boxes'] = grp['boxes'].apply(lambda bl: [[float(v) for v in b.split(',')] for b in bl])
SPLITS = {n: grp[grp.split == k].reset_index(drop=True) for n, k in [('train', 1), ('val', 2), ('test', 3)]}
for k, v in SPLITS.items(): print(f'{k}: {len(v)} images, {sum(map(len, v.boxes))} lesions')

def read_norm(p): return norm(np.clip(load_hu(p), HU_MIN, HU_MAX)).astype(np.float32)

class CTDataset(Dataset):
    def __init__(self, d, train, dose='random', clean_input=False, crop=None):
        self.d, self.train, self.dose, self.clean_input, self.crop = d, train, dose, clean_input, crop
    def __len__(self): return len(self.d)
    def __getitem__(self, i):
        for _ in range(10):                         # an unreadable file is skipped, not fatal
            try: return self._get(i)
            except Exception as e:
                print('skipping unreadable sample', self.d.iloc[i].File_name, e); i = np.random.randint(len(self.d))
        raise RuntimeError('too many unreadable samples — rerun section 4')
    def _get(self, i):
        r = self.d.iloc[i]
        dose = np.random.choice(list(DOSE_LEVELS)) if self.dose == 'random' else self.dose
        y = read_norm(r.Clean)
        x = y.copy() if self.clean_input else read_norm(r.Low if dose == 'low' else r.Mod)
        boxes = np.array(r.boxes, np.float32)
        if self.train and np.random.rand() < 0.5:
            x, y = x[:, ::-1].copy(), y[:, ::-1].copy(); boxes[:, [0, 2]] = 512 - boxes[:, [2, 0]]
        hm, wh, off, ind, mask = make_targets(boxes)
        if self.crop:                                   # random patches (RED-CNN training)
            a, b = np.random.randint(0, 512 - self.crop, 2)
            x, y = x[a:a + self.crop, b:b + self.crop].copy(), y[a:a + self.crop, b:b + self.crop].copy()
        return dict(x=torch.from_numpy(x)[None], y=torch.from_numpy(y)[None], hm=torch.from_numpy(hm)[None],
                    wh=torch.from_numpy(wh), off=torch.from_numpy(off), ind=torch.from_numpy(ind),
                    mask=torch.from_numpy(mask), boxes=boxes, types=list(r.types), name=r.File_name)

def collate(b):
    out = {k: torch.stack([s[k] for s in b]) for k in ('x', 'y', 'hm', 'wh', 'off', 'ind', 'mask')}
    for k in ('boxes', 'types', 'name'): out[k] = [s[k] for s in b]
    return out

def loader(split, train, dose='random', shuffle=None, clean_input=False, crop=None, bs=None):
    return DataLoader(CTDataset(SPLITS[split], train, dose, clean_input, crop), batch_size=bs or BATCH_SIZE,
                      collate_fn=collate, shuffle=train if shuffle is None else shuffle, num_workers=2,
                      pin_memory=True, drop_last=train)

model = build_model(CONFIG).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'{MODEL} | transformer={USE_TRANSFORMER} fusion={USE_FUSION} | {n_params / 1e6:.2f} M params')
b = next(iter(loader('train', True)))
with torch.no_grad(), torch.autocast(device, enabled=device == 'cuda'):
    o = model(b['x'].to(device))
print({k: tuple(v.shape) for k, v in o.items() if torch.is_tensor(v)}, '| fusion maps:', [tuple(a.shape) for a in o['att']])

## 6. Training (checkpoint every epoch, auto-resume)

In [ ]:
json.dump(CONFIG, open(f'{RES_DIR}/config.json', 'w'), indent=1)     # the demo and evaluation read this
is_redcnn = MODEL == 'redcnn'
train_dl = loader('train', True, clean_input=TRAIN_ON_CLEAN, crop=128 if is_redcnn else None, bs=32 if is_redcnn else None)
val_dl = loader('val', False, dose='low', clean_input=TRAIN_ON_CLEAN)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * len(train_dl), pct_start=0.1)
scaler = torch.amp.GradScaler('cuda', enabled=device == 'cuda')
LAST, BEST, HIST = f'{MODEL_DIR}/last.pt', f'{MODEL_DIR}/best.pt', f'{MODEL_DIR}/history.csv'

start, best, history = 0, float('inf'), []
if os.path.exists(LAST):
    ck = torch.load(LAST, map_location=device)
    for obj, key in [(model, 'model'), (opt, 'opt'), (sched, 'sched'), (scaler, 'scaler')]: obj.load_state_dict(ck[key])
    start, best = ck['epoch'] + 1, ck['best']
    history = pd.read_csv(HIST).to_dict('records') if os.path.exists(HIST) else []
    print('Resumed at epoch', start)

def run_epoch(dl, train):
    model.train(train); agg = np.zeros(5); n = 0
    for b in tqdm(dl, leave=False, desc='train' if train else 'val'):
        b = {k: v.to(device, non_blocking=True) if torch.is_tensor(v) else v for k, v in b.items()}
        with torch.set_grad_enabled(train), torch.autocast(device, enabled=device == 'cuda'):
            out = model(b['x'])
            l_den = F.l1_loss(out['den'].float(), b['y'])
            if 'hm' in out and LAMBDA_DET > 0:
                l_det, parts = detection_loss(out, b['hm'], b['wh'], b['off'], b['ind'], b['mask'])
            else:
                l_det, parts = torch.zeros((), device=device), (0., 0., 0.)
            loss = (LAMBDA_DEN if not TRAIN_ON_CLEAN else 0.) * l_den + LAMBDA_DET * l_det
        if train:
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(opt); scaler.update(); sched.step()
        agg += np.array([loss.item(), l_den.item(), *parts]) * len(b['x']); n += len(b['x'])
    return agg / n          # total, den, hm, wh, off

for ep in range(start, EPOCHS):
    t = time.time(); tr = run_epoch(train_dl, True); va = run_epoch(val_dl, False)
    rec = {'epoch': ep, **{f'train_{k}': v for k, v in zip(['loss', 'den', 'hm', 'wh', 'off'], tr)},
           **{f'val_{k}': v for k, v in zip(['loss', 'den', 'hm', 'wh', 'off'], va)}}
    if getattr(model, 'fuse', None) is not None:
        rec.update({f'gate_{i}_{n}': float(getattr(f, n)) for i, f in enumerate(model.fuse) for n in ('g_den', 'g_det')})
    history.append(rec); pd.DataFrame(history).to_csv(HIST, index=False)
    if va[0] < best: best = va[0]; torch.save(model.state_dict(), BEST)
    torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'sched': sched.state_dict(),
                'scaler': scaler.state_dict(), 'epoch': ep, 'best': best}, LAST)
    print(f'ep {ep:3d} | train {tr[0]:.4f} (den {tr[1]:.4f}, hm {tr[2]:.3f}) | val {va[0]:.4f} (den {va[1]:.4f}, hm {va[2]:.3f}) | best {best:.4f} | {time.time() - t:.0f}s')

h = pd.DataFrame(history)
if len(h):
    fig, ax = plt.subplots(1, 3, figsize=(18, 4))
    for a, k, t in zip(ax, ['den', 'hm'], ['Denoising L1', 'Heatmap focal']):
        a.plot(h.epoch, h[f'train_{k}'], label='train'); a.plot(h.epoch, h[f'val_{k}'], label='val'); a.set_title(t); a.set_xlabel('epoch'); a.legend()
    gates = [c for c in h.columns if c.startswith('gate')]
    for c in gates: ax[2].plot(h.epoch, h[c], label=c)
    ax[2].set_title('Cross-task fusion gates' if gates else 'no fusion'); ax[2].set_xlabel('epoch')
    if gates: ax[2].legend()
    plt.tight_layout(); plt.savefig(f'{RES_DIR}/curves.png', dpi=150); plt.show()

## 7. Test-set evaluation
* **Denoising:** PSNR / SSIM (normalized images) and RMSE in HU inside the body, input vs. output, per dose level.
* **Localization:** FROC sensitivity at 0.5–8 FPs/image with IoU ≥ 0.5 (standard DeepLesion protocol), FROC curve, per-lesion-type sensitivity at 4 FPs/image.
* **Efficiency:** parameters, fp16 size, GPU time per 512×512 slice.
* Per-image values and all detections are saved so the significance tests in section 9 can pair runs image by image.

In [ ]:
import pickle
from skimage.metrics import peak_signal_noise_ratio as psnr, structural_similarity as ssim
LESION_TYPES = {1: 'bone', 2: 'abdomen', 3: 'mediastinum', 4: 'liver', 5: 'lung', 6: 'kidney', 7: 'soft_tissue', 8: 'pelvis'}

def evaluate(den_model, det_model, res_dir, det_on_denoised=False):
    # den_model: produces the image metrics (None = skip). det_model: produces detections (None = skip).
    # det_on_denoised: two-stage pipeline — det_model is fed den_model's output.
    results, per_type, curves = {}, {}, {}
    for dose in DOSE_LEVELS:
        rows, dets, gts, tys, names = [], [], [], [], []
        with torch.no_grad():
            for b in tqdm(loader('test', False, dose=dose, bs=4), desc=f'test {dose}'):
                x = b['x'].to(device)
                with torch.autocast(device, enabled=device == 'cuda'):
                    den = den_model(x)['den'].float().clamp(0, 1) if den_model is not None else None
                    if det_model is not None:
                        dets += decode(det_model(den if det_on_denoised else x))
                gts += b['boxes']; tys += b['types']; names += b['name']
                if den is None: continue
                den = den.cpu().numpy()
                for i in range(len(den)):
                    xi, yi, oi = b['x'][i, 0].numpy(), b['y'][i, 0].numpy(), den[i, 0]; body = to_hu(yi) > -500
                    r = {'name': b['name'][i]}
                    for tag, img in (('in', xi), ('out', oi)):
                        r[f'psnr_{tag}'] = psnr(yi, img, data_range=1); r[f'ssim_{tag}'] = ssim(yi, img, data_range=1)
                        r[f'rmseHU_{tag}'] = float(np.sqrt(np.mean((to_hu(img) - to_hu(yi))[body] ** 2)))
                    rows.append(r)
        res = {}
        if rows:
            pi = pd.DataFrame(rows); pi.to_csv(f'{res_dir}/per_image_{dose}.csv', index=False)
            res.update(pi.drop(columns='name').mean().to_dict())
        if dets:
            pickle.dump({'names': names, 'dets': dets, 'gts': gts, 'types': tys}, open(f'{res_dir}/detections_{dose}.pkl', 'wb'))
            recs, les = match_detections(dets, gts)
            nl = sum(map(len, gts))
            fr, thr = froc(recs, len(gts), nl)
            res.update(fr); res['mean_sens'] = float(np.mean(list(fr.values())))
            xs, ys = froc_curve(recs, len(gts), nl)
            pd.DataFrame({'fp_per_image': xs, 'sensitivity': ys}).to_csv(f'{res_dir}/froc_curve_{dose}.csv', index=False)
            lt = pd.DataFrame({'type': [t for ts in tys for t in ts], 'score': [s for ls in les for s in ls]})
            lt = lt[lt.type > 0]
            per_type[dose] = lt.assign(hit=lt.score >= thr[4]).groupby('type').hit.mean().rename(index=LESION_TYPES)
        results[dose] = res
    return pd.DataFrame(results).T, pd.DataFrame(per_type)

def efficiency(m):
    x = torch.randn(1, 1, 512, 512, device=device)
    with torch.no_grad(), torch.autocast(device, enabled=device == 'cuda'):
        for _ in range(5): m(x)
        if device == 'cuda': torch.cuda.synchronize()
        t = time.time()
        for _ in range(20): m(x)
        if device == 'cuda': torch.cuda.synchronize()
    p = sum(q.numel() for q in m.parameters())
    return {'params_M': p / 1e6, 'size_fp16_MB': p * 2 / 2 ** 20, 'ms_per_slice': (time.time() - t) / 20 * 1000}

model.load_state_dict(torch.load(BEST, map_location=device)); model.eval()
has_det = MODEL == 'ctfusion' and LAMBDA_DET > 0
has_den = LAMBDA_DEN > 0 and not TRAIN_ON_CLEAN
res_df, type_df = evaluate(model if has_den else None, model if has_det else None, RES_DIR)
for k, v in efficiency(model).items(): res_df[k] = v
res_df.round(4).to_csv(f'{RES_DIR}/test_metrics.csv')
if len(type_df): type_df.round(3).to_csv(f'{RES_DIR}/per_type_sensitivity.csv')
print(res_df.round(4).T.to_string())
if len(type_df): print('\nSensitivity @4 FP/image by lesion type:\n', type_df.round(3).to_string())

## 8. Qualitative results

In [ ]:
if MODEL == 'ctfusion' and has_det and has_den:
    b = next(iter(loader('test', False, dose='low', shuffle=True)))
    with torch.no_grad(), torch.autocast(device, enabled=device == 'cuda'):
        out = model(b['x'].to(device))
    den = out['den'].float().clamp(0, 1).cpu().numpy(); dets = decode(out, thr=0.3)
    N = min(4, len(den)); cols = 5 if out['att'] else 4
    fig, ax = plt.subplots(N, cols, figsize=(4.2 * cols, 4.2 * N)); ax = np.atleast_2d(ax)
    for i in range(N):
        imgs = [(to_hu(b['x'][i, 0].numpy()), f'Low-dose\n{b["name"][i]}'), (to_hu(den[i, 0]), 'CTFusionNet output'),
                (to_hu(b['y'][i, 0].numpy()), 'Original'), (to_hu(den[i, 0]), 'Pred (red) / GT (green)')]
        for j, (img, t) in enumerate(imgs):
            ax[i, j].imshow(img, cmap='gray', vmin=-160, vmax=240); ax[i, j].set_title(t); ax[i, j].axis('off')
        for x1, y1, x2, y2 in b['boxes'][i]: ax[i, 3].add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec='lime', lw=2))
        for s, (x1, y1, x2, y2) in dets[i][:3]:
            ax[i, 3].add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec='red', lw=1.5)); ax[i, 3].text(x1, y1 - 4, f'{s:.2f}', color='red')
        if out['att']:
            ax[i, 4].imshow(to_hu(den[i, 0]), cmap='gray', vmin=-160, vmax=240)
            ax[i, 4].imshow(cv2.resize(out['att'][-1][i, 0].float().cpu().numpy(), (512, 512)), cmap='jet', alpha=0.45)
            ax[i, 4].set_title('Fusion lesion attention'); ax[i, 4].axis('off')
    plt.tight_layout(); plt.savefig(f'{RES_DIR}/qualitative.png', dpi=150); plt.show()
else:
    print('Qualitative grid is drawn for the full multitask model only.')

## 9. Experiments, baselines, statistics and paper figures

Set `EXPERIMENT`, `SEED` and the flags in section 0, then **Run all** (data stages are skipped automatically).

| EXPERIMENT | MODEL | USE_TRANSFORMER | USE_FUSION | LAMBDA_DEN | LAMBDA_DET | TRAIN_ON_CLEAN | Seeds | Purpose |
|---|---|---|---|---|---|---|---|---|
| `ctfusion_full` | ctfusion | True | True | 1 | 1 | False | 42, 43, 44 | **proposed model** |
| `no_fusion` | ctfusion | True | False | 1 | 1 | False | 42, 43, 44 | does cross-task fusion help? |
| `no_transformer` | ctfusion | False | True | 1 | 1 | False | 42 | does global context help? |
| `cnn_baseline` | ctfusion | False | False | 1 | 1 | False | 42 | plain multitask CNN |
| `denoise_only` | ctfusion | True | False | 1 | 0 | False | 42 | single-task denoising |
| `detect_only` | ctfusion | True | False | 0 | 1 | False | 42 | single-task localization on low-dose input |
| `redcnn` | redcnn | – | – | 1 | 0 | False | 42 | published denoiser (RED-CNN, Chen et al. 2017) |
| `detect_clean` | ctfusion | True | False | 0 | 1 | **True** | 42 | detector trained on normal-dose images |
| `two_stage` | – | – | – | – | – | – | 42 | **RED-CNN → detect_clean** (run the next cell once both exist) |

Single-task runs report only the metrics of their task.

In [ ]:
# ---- two-stage baseline: RED-CNN denoising followed by a detector trained on normal-dose images ----
den_dir, det_dir = f'{PROJECT}/models/redcnn_s{SEED}', f'{PROJECT}/models/detect_clean_s{SEED}'
if os.path.exists(f'{den_dir}/best.pt') and os.path.exists(f'{det_dir}/best.pt'):
    load = lambda d: json.load(open(d.replace('/models/', '/results/') + '/config.json'))
    den_m, det_m = build_model(load(den_dir)).to(device), build_model(load(det_dir)).to(device)
    den_m.load_state_dict(torch.load(f'{den_dir}/best.pt', map_location=device)); den_m.eval()
    det_m.load_state_dict(torch.load(f'{det_dir}/best.pt', map_location=device)); det_m.eval()
    ts_dir = f'{PROJECT}/results/two_stage_s{SEED}'; os.makedirs(ts_dir, exist_ok=True)
    ts_res, ts_type = evaluate(den_m, det_m, ts_dir, det_on_denoised=True)
    e1, e2 = efficiency(den_m), efficiency(det_m)
    for k in e1: ts_res[k] = e1[k] + e2[k]
    ts_res.round(4).to_csv(f'{ts_dir}/test_metrics.csv'); ts_type.round(3).to_csv(f'{ts_dir}/per_type_sensitivity.csv')
    print(ts_res.round(4).T.to_string())
else:
    print('Two-stage baseline needs both redcnn and detect_clean trained for seed', SEED)

In [ ]:
# ---- summary table (mean ± std over seeds) ----
rows = []
for f in sorted(glob.glob(f'{PROJECT}/results/*_s*/test_metrics.csv')):
    run = f.split('/')[-2]
    t = pd.read_csv(f, index_col=0).reset_index().rename(columns={'index': 'dose'})
    t.insert(0, 'experiment', re.sub(r'_s\d+$', '', run)); t.insert(1, 'seed', run.split('_s')[-1]); rows.append(t)
if rows:
    allr = pd.concat(rows); allr.to_csv(f'{PROJECT}/results/all_runs.csv', index=False)
    cols = [c for c in ['psnr_in', 'psnr_out', 'ssim_out', 'rmseHU_out', 'sens@0.5FP', 'sens@1FP', 'sens@2FP', 'sens@4FP',
                        'sens@8FP', 'mean_sens', 'params_M', 'ms_per_slice'] if c in allr]
    g = allr.groupby(['experiment', 'dose'])[cols]
    mean, std = g.mean(), g.std().fillna(0)
    table = (mean.round(4).astype(str) + ' ± ' + std.round(4).astype(str)).mask(mean.isna(), '–')
    table = table.join(g.size().rename('n_seeds'))
    table.to_csv(f'{PROJECT}/results/summary_table.csv')
    try: open(f'{PROJECT}/results/summary_table.tex', 'w').write(mean.round(3).to_latex(na_rep='–'))
    except Exception as e: print('LaTeX export skipped:', e)
    print(table.to_string())
else:
    print('No finished runs yet.')

In [ ]:
# ---- significance tests: proposed model vs. each comparison (paired by test image, averaged over seeds) ----
from scipy.stats import wilcoxon
PROPOSED = 'ctfusion_full'
def runs_of(exp): return sorted(glob.glob(f'{PROJECT}/results/{exp}_s*'))

def per_image(exp, dose):
    fs = [f'{r}/per_image_{dose}.csv' for r in runs_of(exp) if os.path.exists(f'{r}/per_image_{dose}.csv')]
    return pd.concat([pd.read_csv(f) for f in fs]).groupby('name').mean() if fs else None

def boot_sens(exp, dose, idx_names, B=1000, seed=0):
    # bootstrap distribution of mean FROC sensitivity, resampling test images (seeds pooled by averaging)
    packs = [pickle.load(open(f'{r}/detections_{dose}.pkl', 'rb')) for r in runs_of(exp)
             if os.path.exists(f'{r}/detections_{dose}.pkl')]
    if not packs: return None
    per = []                                         # per run: per-image match records + lesion counts, in idx_names order
    for p in packs:
        pos = {n: i for i, n in enumerate(p['names'])}
        per.append([(match_detections([p['dets'][pos[n]]], [p['gts'][pos[n]]])[0], len(p['gts'][pos[n]])) for n in idx_names])
    rng, out = np.random.default_rng(seed), []
    for _ in range(B):
        sel = rng.integers(0, len(idx_names), len(idx_names)); vals = []
        for img in per:
            recs = [r for j in sel for r in img[j][0]]
            fr, _ = froc(recs, len(sel), sum(img[j][1] for j in sel)); vals.append(np.mean(list(fr.values())))
        out.append(np.mean(vals))
    return np.array(out)

exps = sorted({re.sub(r'_s\d+$', '', os.path.basename(r)) for r in glob.glob(f'{PROJECT}/results/*_s*')})
sig = []
for dose in DOSE_LEVELS:
    base = per_image(PROPOSED, dose)
    for exp in [e for e in exps if e != PROPOSED]:
        row = {'dose': dose, 'comparison': f'{PROPOSED} vs {exp}'}
        other = per_image(exp, dose)
        if base is not None and other is not None:
            j = base.join(other, lsuffix='_a', rsuffix='_b', how='inner')
            for m in ('psnr_out', 'ssim_out'):
                row[f'{m}_diff'] = float((j[f'{m}_a'] - j[f'{m}_b']).mean())
                row[f'{m}_p'] = float(wilcoxon(j[f'{m}_a'], j[f'{m}_b']).pvalue)
        names = SPLITS['test'].File_name.tolist()
        a = boot_sens(PROPOSED, dose, names, B=500); c = boot_sens(exp, dose, names, B=500) if a is not None else None
        if a is not None and c is not None:
            d = a - c
            row['mean_sens_diff'] = float(d.mean()); row['mean_sens_95CI'] = f'[{np.percentile(d, 2.5):.3f}, {np.percentile(d, 97.5):.3f}]'
        sig.append(row)
if sig:
    sig = pd.DataFrame(sig); sig.to_csv(f'{PROJECT}/results/significance.csv', index=False); print(sig.round(4).to_string(index=False))
    print('\npsnr/ssim: Wilcoxon signed-rank p-values. Detection: bootstrap 95% CI of the sensitivity difference (excludes 0 → significant).')

In [ ]:
# ---- paper figures (saved as PNG at 300 dpi in results/figures) ----
FIG = f'{PROJECT}/results/figures'
plt.rcParams.update({'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False})
if not os.path.exists(f'{PROJECT}/results/all_runs.csv'):
    print('Run the summary cell first.')
else:
    allr = pd.read_csv(f'{PROJECT}/results/all_runs.csv')
    order = [e for e in ['ctfusion_full', 'no_fusion', 'no_transformer', 'cnn_baseline', 'denoise_only', 'detect_only',
                         'redcnn', 'two_stage'] if e in set(allr.experiment)]

    # 1. FROC curves (low dose), mean over seeds
    plt.figure(figsize=(6, 4.5))
    for exp in order:
        fs = glob.glob(f'{PROJECT}/results/{exp}_s*/froc_curve_low.csv')
        if fs:
            c = pd.concat([pd.read_csv(f) for f in fs]).groupby('fp_per_image').sensitivity.mean()
            plt.plot(c.index, c.values, lw=2.5 if exp == 'ctfusion_full' else 1.4, label=exp)
    plt.xscale('log', base=2); plt.xticks([0.125, 0.25, 0.5, 1, 2, 4, 8], ['1/8', '1/4', '1/2', '1', '2', '4', '8'])
    plt.xlabel('False positives per image'); plt.ylabel('Sensitivity (IoU ≥ 0.5)'); plt.title('FROC — low dose'); plt.legend(fontsize=9)
    plt.grid(alpha=0.3); plt.tight_layout(); plt.savefig(f'{FIG}/froc_low.png', dpi=300); plt.show()

    # 2. Ablation bars (low dose): PSNR and mean sensitivity, mean ± std over seeds
    low = allr[allr.dose == 'low']
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
    for a, m, t in zip(ax, ['psnr_out', 'mean_sens'], ['Denoising PSNR (dB)', 'Mean FROC sensitivity']):
        if m not in low: continue
        g = low.groupby('experiment')[m].agg(['mean', 'std']).reindex(order).dropna(subset=['mean'])
        a.bar(g.index, g['mean'], yerr=g['std'].fillna(0), capsize=4,
              color=['#c0392b' if e == 'ctfusion_full' else '#7f8c8d' for e in g.index])
        if m == 'psnr_out' and 'psnr_in' in low:
            a.axhline(low.psnr_in.mean(), ls='--', c='k', lw=1, label='low-dose input'); a.legend()
        lo = (g['mean'] - g['std'].fillna(0)).min()
        if m == 'psnr_out' and 'psnr_in' in low: lo = min(lo, low.psnr_in.mean())
        a.set_ylim(lo - abs(lo) * 0.05 if m == 'psnr_out' else 0, None); a.set_title(t); a.tick_params(axis='x', rotation=35)
    plt.tight_layout(); plt.savefig(f'{FIG}/ablation_bars.png', dpi=300); plt.show()

    # 3. Performance vs dose
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
    dose_order = list(DOSE_LEVELS)
    for a, m, t in zip(ax, ['psnr_out', 'mean_sens'], ['PSNR (dB)', 'Mean FROC sensitivity']):
        if m not in allr: continue
        for exp in order:
            g = allr[allr.experiment == exp].groupby('dose')[m].mean().reindex(dose_order)
            if g.notna().any(): a.plot([f'{d}\n(I0={DOSE_LEVELS[d]:.0e})' for d in dose_order], g.values, 'o-', label=exp)
        a.set_title(t)
    ax[0].legend(fontsize=8); plt.tight_layout(); plt.savefig(f'{FIG}/performance_vs_dose.png', dpi=300); plt.show()

    # 4. Per-lesion-type sensitivity (low dose): proposed vs no fusion
    fig, a = plt.subplots(figsize=(10, 4.5)); w = 0.38
    for k, exp in enumerate([e for e in ['ctfusion_full', 'no_fusion', 'two_stage'] if e in order]):
        fs = glob.glob(f'{PROJECT}/results/{exp}_s*/per_type_sensitivity.csv')
        if not fs: continue
        t = pd.concat([pd.read_csv(f, index_col=0)['low'] for f in fs], axis=1).mean(axis=1)
        a.bar(np.arange(len(t)) + k * w / 1.5, t.values, w / 1.5, label=exp)
        a.set_xticks(np.arange(len(t)) + w / 2); a.set_xticklabels(t.index, rotation=30)
    a.set_ylabel('Sensitivity @ 4 FP/image'); a.set_title('Per-lesion-type sensitivity — low dose'); a.legend()
    plt.tight_layout(); plt.savefig(f'{FIG}/per_type_sensitivity.png', dpi=300); plt.show()
    print('Figures saved to', FIG)

## 10. Streamlit demo
Starts the app and opens a public `trycloudflare.com` link (no account needed). Works on Colab and on Kaggle (Kaggle: Internet must be on).
Pick a DeepLesion test slice or upload one, choose the dose (pre-generated or a custom I₀ slider), and see the denoised image,
detections vs. ground truth, PSNR/SSIM, inference time and the cross-task fusion attention maps. Keep the notebook running during the demo.
**Demo only (model already trained):** run section 0 (both cells), 3a (both cells) and this section.

In [ ]:
%%writefile app.py
import os, json, time
import numpy as np, pandas as pd, cv2, torch, streamlit as st
from skimage.metrics import peak_signal_noise_ratio as psnr, structural_similarity as ssim
from ctfusion import *

PROJECT = os.environ.get('CT_PROJECT', '/content/drive/MyDrive/CT_Multitask_Research')
RUN     = os.environ.get('CT_RUN', 'ctfusion_full')
LESION_TYPES = {1: 'Bone', 2: 'Abdomen', 3: 'Mediastinum', 4: 'Liver', 5: 'Lung', 6: 'Kidney', 7: 'Soft tissue', 8: 'Pelvis'}
st.set_page_config(page_title='CTFusionNet demo', layout='wide')

@st.cache_resource
def load_model():
    cfg = json.load(open(f'{PROJECT}/results/{RUN}/config.json'))
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    m = build_model(cfg)
    m.load_state_dict(torch.load(f'{PROJECT}/models/{RUN}/best.pt', map_location=dev))
    return m.to(dev).eval(), cfg, dev

@st.cache_data
def load_test():
    df = pd.read_csv(f'{PROJECT}/data/splits/manifest_paired.csv')
    return df[df['Train_Val_Test'] == 3]

@st.cache_data(show_spinner='Simulating low-dose scan (Radon → Poisson → FBP)...')
def simulate(key, _hu, spacing, I0, n_angles, mu_water):
    return simulate_low_dose(_hu, spacing, I0, seed=0, n_angles=n_angles, mu_water=mu_water)

model, cfg, dev = load_model()
test = load_test()
doses = cfg['DOSE_LEVELS']

st.title('CTFusionNet — low-dose CT denoising + lesion localization')
st.caption(f"Run `{RUN}` · model: {cfg.get('MODEL', 'ctfusion')} · transformer: {cfg['USE_TRANSFORMER']} · cross-task fusion: {cfg['USE_FUSION']} · "
           f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f} M parameters · device: {dev}")

with st.sidebar:
    st.header('Input')
    src = st.radio('Image source', ['DeepLesion test set', 'Upload PNG'])
    gt, types, ld = [], [], None
    if src == 'DeepLesion test set':
        name = st.selectbox('Test slice', sorted(test['File_name'].unique()))
        rows = test[test['File_name'] == name]
        clean = np.clip(load_hu(rows.iloc[0]['Image_Path']), HU_MIN, HU_MAX)
        spacing = float(str(rows.iloc[0]['Spacing_mm_px_']).split(',')[0])
        gt = [[float(v) for v in b.split(',')] for b in rows['Bounding_boxes']]
        types = rows['Coarse_lesion_type'].tolist()
        options = [f'Pre-generated {d} dose (I0={v:.0e})' for d, v in doses.items()] + ['Custom simulation']
    else:
        up = st.file_uploader('DeepLesion-style 16-bit PNG (HU + 32768)', type=['png'])
        if up is None:
            st.info('Upload a slice to start.'); st.stop()
        name = up.name
        clean, is16 = hu_from_png_bytes(up.read())
        if not is16: st.warning('8-bit image: HU approximated from a soft-tissue window.')
        if clean.shape != (512, 512): clean = cv2.resize(clean, (512, 512), interpolation=cv2.INTER_AREA)
        clean = np.clip(clean, HU_MIN, HU_MAX)
        spacing = st.number_input('Pixel spacing (mm)', 0.3, 1.5, 0.7, 0.01)
        options = ['Custom simulation', 'Image is already low-dose (no simulation)']
    choice = st.selectbox('Low-dose input', options)
    if choice == 'Custom simulation':
        I0 = 10 ** st.slider('Dose level: log10(I0)  (lower = noisier)', 4.5, 7.0, 6.0, 0.25)
    st.header('Display')
    thr = st.slider('Detection score threshold', 0.05, 0.95, 0.30, 0.05)
    win = WINDOWS[st.selectbox('Window', list(WINDOWS))]
    show_att = st.checkbox('Show cross-task lesion attention', value=cfg['USE_FUSION'])

has_ref = True
if choice.startswith('Pre-generated'):
    d = choice.split()[1]
    ld = np.clip(load_hu(rows.iloc[0]['LowDose_Path' if d == 'low' else 'ModDose_Path']), HU_MIN, HU_MAX)
elif choice == 'Custom simulation':
    ld = np.clip(simulate(f'{name}-{I0}', clean, spacing, I0, cfg['N_ANGLES'], cfg['MU_WATER']), HU_MIN, HU_MAX)
else:
    ld, has_ref = clean.copy(), False

x = torch.from_numpy(norm(ld).astype(np.float32))[None, None].to(dev)
t0 = time.time()
with torch.no_grad():
    out = model(x)
if dev == 'cuda': torch.cuda.synchronize()
ms = (time.time() - t0) * 1000
den = to_hu(out['den'][0, 0].float().clamp(0, 1).cpu().numpy())
dets = decode(out, K=10, thr=thr)[0] if 'hm' in out else []

def overlay(hu_img, boxes_pred, boxes_gt):
    rgb = cv2.cvtColor(window(hu_img, win), cv2.COLOR_GRAY2RGB)
    for b in boxes_gt:
        x1, y1, x2, y2 = map(int, b); cv2.rectangle(rgb, (x1, y1), (x2, y2), (0, 255, 0), 2)
    for s, b in boxes_pred:
        x1, y1, x2, y2 = map(int, b); cv2.rectangle(rgb, (x1, y1), (x2, y2), (255, 40, 40), 2)
        cv2.putText(rgb, f'{s:.2f}', (x1, max(y1 - 4, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 40, 40), 1)
    return rgb

c = st.columns(4)
c[0].image(window(ld, win), caption='Low-dose input')
c[1].image(window(den, win), caption=f'Denoised output ({ms:.0f} ms)')
c[2].image(window(clean, win), caption='Reference (original scan)' if has_ref else 'Input')
c[3].image(overlay(den, dets, gt), caption='Detections (red) vs ground truth (green)')

if has_ref:
    y, xin, xo = norm(clean), norm(ld), norm(den)
    body = clean > -500
    m = st.columns(4)
    p_in, p_out = psnr(y, xin, data_range=1), psnr(y, xo, data_range=1)
    s_in, s_out = ssim(y, xin, data_range=1), ssim(y, xo, data_range=1)
    m[0].metric('PSNR (dB)', f'{p_out:.2f}', f'{p_out - p_in:+.2f} vs input')
    m[1].metric('SSIM', f'{s_out:.4f}', f'{s_out - s_in:+.4f} vs input')
    m[2].metric('Noise std in body (HU)', f'{(den - clean)[body].std():.1f}', f'{(den - clean)[body].std() - (ld - clean)[body].std():+.1f}', delta_color='inverse')
    m[3].metric('Lesions detected', f'{sum(any(iou(b, [g])[0] >= 0.5 for _, b in dets) for g in gt)} / {len(gt)}' if gt else '—')

if show_att and out['att']:
    st.subheader('Cross-task fusion: lesion attention passed from the localization branch to the denoiser')
    a = st.columns(len(out['att']))
    for i, att in enumerate(out['att']):
        amap = cv2.resize(att[0, 0].float().cpu().numpy(), (512, 512))
        heat = cv2.applyColorMap((amap / max(amap.max(), 1e-6) * 255).astype(np.uint8), cv2.COLORMAP_JET)[..., ::-1]
        blend = cv2.addWeighted(cv2.cvtColor(window(den, win), cv2.COLOR_GRAY2RGB), 0.6, heat, 0.4, 0)
        a[i].image(blend, caption=f'Fusion stage {i + 1} ({att.shape[-1]}×{att.shape[-1]})')

st.subheader('Detections')
if dets:
    st.dataframe(pd.DataFrame([{'score': round(s, 3), 'x1': round(b[0]), 'y1': round(b[1]), 'x2': round(b[2]), 'y2': round(b[3]),
                                'width (mm)': round((b[2] - b[0]) * spacing, 1), 'height (mm)': round((b[3] - b[1]) * spacing, 1),
                                'best IoU with GT': round(float(iou(b, gt).max()), 2) if gt else None} for s, b in dets]))
else:
    st.write('No detections above the threshold.')
if types:
    st.caption('Ground-truth lesion types: ' + ', '.join(LESION_TYPES.get(int(t), 'unknown') for t in types))
st.caption('Low-dose images are simulated from DeepLesion (Poisson noise in the projection domain + FBP). Research prototype, not for clinical use.')

In [ ]:
!pip -q install streamlit
CF = f'{SCRATCH}/cloudflared'
if not os.path.exists(CF):
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', CF)
    os.chmod(CF, 0o755)
assert os.path.exists(f'{PROJECT}/models/{RUN_NAME}/best.pt'), f'Train {RUN_NAME} first (section 6)'
shutil.copy('app.py', f'{PROJECT}/code/app.py')
os.environ.update(CT_PROJECT=PROJECT, CT_RUN=RUN_NAME, PYTHONPATH=os.getcwd())
subprocess.run('pkill -f "streamlit run"; pkill -f cloudflared', shell=True)
subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', '8501', '--server.headless', 'true'],
                 cwd=os.getcwd(), stdout=open(f'{SCRATCH}/streamlit.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(5)
subprocess.Popen([CF, 'tunnel', '--url', 'http://localhost:8501', '--no-autoupdate'],
                 stdout=open(f'{SCRATCH}/cloudflared.log', 'w'), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', open(f'{SCRATCH}/cloudflared.log').read())
    if m: url = m.group(0); break
print('Demo link:', url if url else f'not ready, check {SCRATCH}/cloudflared.log')